# Q1Net: drift-robust JPEG-quality classifier (Kaggle GPU)

End-to-end: clone repo (`drift-aug` branch) -> DIV2K download -> HDF5 generation
(24x24 grid-aligned patches) -> fine-tune 45 epochs with random sub-block drift
crops -> compare against the committed baseline checkpoint on aligned and
misaligned (off-8x8-grid cropped) images.

**Setup**: Settings > Accelerator = GPU (T4/P100), Internet = on.

In [ ]:
# 1. Environment: Keras 2 shim for the TF preinstalled on Kaggle
import os, importlib.metadata
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf_minor = '.'.join(importlib.metadata.version('tensorflow').split('.')[:2])
!pip install -q "tf-keras=={tf_minor}.*" || pip install -q tf-keras
!git lfs version || (apt-get update -qq && apt-get install -y -qq git-lfs)
!nvidia-smi -L

In [ ]:
# 2. Clone the public repo (drift-aug branch) and pull the LFS baseline weights
import os, sys
os.makedirs('/kaggle/tmp', exist_ok=True)
%cd /kaggle/tmp
!rm -rf Q1Net
!git clone --branch drift-aug --depth 1 https://github.com/chammoru/Q1Net.git
%cd Q1Net/classifier
!git lfs install --skip-repo && git lfs pull
os.environ['PYTHONPATH'] = os.path.abspath('..') + ':' + os.path.abspath('.')
sys.path.insert(0, os.path.abspath('.')); sys.path.insert(0, os.path.abspath('..'))
import class_core
class_core.ensure_weights_available('./save/jpeg_paper/best/')
print('baseline weights OK')

In [ ]:
# 3. DIV2K sources (800 train + 100 valid HR images)
!wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip && unzip -q DIV2K_train_HR.zip && rm DIV2K_train_HR.zip
!wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip && unzip -q DIV2K_valid_HR.zip && rm DIV2K_valid_HR.zip
print(len(os.listdir('DIV2K_train_HR')), 'train /', len(os.listdir('DIV2K_valid_HR')), 'valid images')

In [ ]:
# 4. Generate 24x24 grid-aligned patch datasets (drift crops happen at train time)
#    train: 800 imgs x 10 patches x 100 qualities = 800k samples (~1.4 GB)
#    val:   100 imgs x  2 patches x 100 qualities =  20k samples
!python gen_data.py --in_path ./DIV2K_train_HR --num_samples 8000 --hdf5_name cls_train_drift.h5 --comp_type jpeg_paper
!python gen_data.py --in_path ./DIV2K_valid_HR --num_samples 200 --hdf5_name cls_val_drift.h5 --comp_type jpeg_paper
!ls -la *.h5

In [ ]:
# 5. Baseline: committed checkpoint on drifted val patches and misaligned images
import numpy as np, cv2, glob
import util
from predict_cls import predict_quality

QUALITIES = list(range(5, 101, 10))
IMAGES = sorted(glob.glob('DIV2K_valid_HR/*.png'))[:15]

def image_mae(weights_dir, off_h, off_w):
    config = class_core.get_classifier_config('jpeg_paper')
    model = class_core.build_model(config, trainable=False)
    model.load_weights(weights_dir).expect_partial()
    errs = []
    for path in IMAGES:
        img = cv2.imread(path)
        for q in QUALITIES:
            comp = util.generate_jpeg(img, q)[off_h:, off_w:]
            errs.append(abs(q - predict_quality(model, comp, config)))
    return float(np.mean(errs))

!cp -r save/jpeg_paper/best save/jpeg_paper/baseline_best
base_aligned = image_mae('./save/jpeg_paper/baseline_best/', 0, 0)
base_drift = image_mae('./save/jpeg_paper/baseline_best/', 3, 5)
print(f'BASELINE image-level MAE: aligned {base_aligned:.2f} | misaligned(3,5) {base_drift:.2f}')
print('BASELINE patch-level (drifted val):')
!python evaluate_cls.py --hdf5_val_path cls_val_drift.h5 --comp_type jpeg_paper 2>/dev/null | tail -2

In [ ]:
# 6. Fine-tune from the baseline with random drift crops (45 epochs)
EPOCHS = 45
!rm -f save/jpeg_paper/best/val_loss.txt stop_training
!python train.py --hdf5_train_path cls_train_drift.h5 --hdf5_val_path cls_val_drift.h5 --comp_type jpeg_paper --epochs {EPOCHS} --batch_size 1024 --lr 3e-4 --verbose 2 --base_weights ./save/jpeg_paper/baseline_best/
import os
assert os.path.exists('save/jpeg_paper/best/val_loss.txt'), 'train.py did not save a new best checkpoint - it probably crashed; check the cell output above'
print('training saved a new best; val_loss =', open('save/jpeg_paper/best/val_loss.txt').read())

In [ ]:
# 7. Compare baseline vs new best
new_aligned = image_mae('./save/jpeg_paper/best/', 0, 0)
new_drift = image_mae('./save/jpeg_paper/best/', 3, 5)
print(f'BASELINE image MAE: aligned {base_aligned:.2f} | misaligned(3,5) {base_drift:.2f}')
print(f'NEW      image MAE: aligned {new_aligned:.2f} | misaligned(3,5) {new_drift:.2f}')
print('NEW patch-level (drifted val):')
!python evaluate_cls.py --hdf5_val_path cls_val_drift.h5 --comp_type jpeg_paper 2>/dev/null | tail -2
for off in [(1, 1), (2, 6), (4, 4), (7, 3)]:
    print('new model, crop offset', off, '-> image MAE', round(image_mae('./save/jpeg_paper/best/', *off), 2))

In [ ]:
# 8. Persist results to /kaggle/working (notebook output)
!mkdir -p /kaggle/working/save
!cp -r save/jpeg_paper /kaggle/working/save/
!cp -r save/last /kaggle/working/save/ 2>/dev/null || true
!cp -r out /kaggle/working/ 2>/dev/null || true
!find /kaggle/working -type f | head -30